# 30 corridas con la corrección del cierre, cuaderno A

En las órdenes que empiezan al cierre, el sistema compraba como máximo el 93,75 % de la orden: el Maestro tiene solo 4 decisiones y cada una asigna como mucho la mitad de lo que falta. La corrección hace que **la última decisión del día asigne todo lo que falta** (`--completar-al-cierre`).

Este cuaderno repite **solo el sistema final** en MO-01, MO-02, MO-03, MO-04, MO-05 × 3 semillas, con las mismas redes, semillas e impacto. TWAP y VWAP **no se repiten**: la corrección no los cambia y se reutilizan los de `corridas_30`.

**Cómo usarlo:** corre los cuadernos A y B **al mismo tiempo**, con «Ejecutar todas». Cada uno dura ≈ **1 hora**. Si se corta, vuelve a ejecutar todo y retoma donde quedó. El que termine último arma la tabla con las 30 corridas y `corridas_30_completa.zip`: ese es el zip que hay que mandar.

## Preparación del entorno (celdas 1 a 9)

In [ ]:
# CELDA 1 (reinicia el entorno solo; el aviso "sesión fallada" es normal, no la repitas)
!pip install -q "condacolab<0.2"
import condacolab
condacolab.install()

In [ ]:
# CELDA 2
import os
if not os.path.exists('/usr/local/envs/abides'):
    !conda create -n abides python=3.9 -y -q
!conda run -n abides python --version

In [ ]:
# CELDA 3 (unos minutos; un aviso de Pillow es normal)
%env PIP_DISABLE_PIP_VERSION_CHECK=1
!conda run -n abides python -m pip install -q pip==23.3.2 setuptools==57.5.0 wheel==0.38.4
!conda run -n abides python -m pip install -q "cython<3" numpy==1.22.0
!conda run -n abides python -m pip install -q --no-build-isolation --no-deps gym==0.18.0
!conda run -n abides python -m pip install -q --no-build-isolation numpy==1.22.0 "cloudpickle>=1.2,<1.7" "pyglet>=1.4,<=1.5.0" "Pillow<10" pandas==1.2.4 scipy==1.10.0 coloredlogs==15.0.1 psutil==5.8.0 tqdm==4.61.1 pomegranate==0.14.5 "ray[tune]==1.7.0" "protobuf==3.20.3" pyarrow==11.0.0 matplotlib==3.5.3 pytest
!conda run -n abides python -m pip install -q "gymnasium==0.29.1" "importlib-metadata>=4.8" farama-notifications --no-deps

In [ ]:
# CELDA 4 (deben salir tres "OK")
import os
if not os.path.exists('/content/abides-jpmc-public'):
    !git clone -q https://github.com/jpmorganchase/abides-jpmc-public.git /content/abides-jpmc-public
for pkg in ['abides-core', 'abides-markets', 'abides-gym']:
    !cd /content/abides-jpmc-public/{pkg} && conda run -n abides python setup.py -q install > /content/install_{pkg}.log 2>&1 && echo "{pkg} OK" || (echo "{pkg} FALLO"; tail -20 /content/install_{pkg}.log)

In [ ]:
# CELDA 5 (debe imprimir 0.18.0 1.22.0 1.2.4 0.29.1 1.7.0)
!conda run -n abides python -c "import gym, numpy, pandas, gymnasium, ray, abides_core, abides_markets, abides_gym; print(gym.__version__, numpy.__version__, pandas.__version__, gymnasium.__version__, ray.__version__)"

In [ ]:
# CELDA 6 (torch; debe imprimir torch 1.13.1+cpu | numpy 1.22.0)
!conda run -n abides python -m pip install -q torch==1.13.1+cpu --extra-index-url https://download.pytorch.org/whl/cpu
!conda run -n abides python -c "import torch, numpy; print('torch', torch.__version__, '| numpy', numpy.__version__)"

In [ ]:
# CELDA 7 (código del proyecto: rama feature/ps-sprint6-final, o main si ya se mergeó; debe mostrar un commit y un número mayor que 0)
import os
if not os.path.exists('/content/slippage'):
    !git clone -q https://github.com/Vatelfil/slippage.git /content/slippage
!cd /content/slippage && git fetch -q origin && (git checkout -q -f origin/feature/ps-sprint6-final || git checkout -q -f origin/main)
!cd /content/slippage && git log --oneline -1 && grep -c "completar_al_cierre" src/envs/maestro_ejecutor_protocol.py


In [ ]:
# CELDA 8 (conecta Drive; acepta el permiso)
from google.colab import drive
drive.mount('/content/drive')
RESULTS_DIR = '/content/drive/MyDrive/slippage_colab/resultados'
import os; os.makedirs(RESULTS_DIR, exist_ok=True)
print(RESULTS_DIR)

In [ ]:
# CELDA 9 (seaborn, que necesita el módulo de benchmarks)
!conda run -n abides python -m pip install -q seaborn==0.12.2 --no-deps

## Sistema final con la corrección del cierre

In [ ]:
!cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/colab/evaluar_sistema_ps.py --meta-ordenes data/meta_ordenes/meta_ordenes_v2_candidata.json --entorno abides --ordenes MO-01 MO-02 MO-03 MO-04 MO-05 --seeds 3 --impacto --completar-al-cierre --out-dir {RESULTS_DIR}/corridas_30_completa/A --max-minutes 330 --modo muestreo --ckpt-maestro {RESULTS_DIR}/sistema_final/maestro/checkpoint_last.pt --ckpt-ejecutor apertura={RESULTS_DIR}/sistema_final/maestro/ejecutor_apertura.pt --ckpt-ejecutor media_jornada={RESULTS_DIR}/sistema_final/maestro/ejecutor_media_jornada.pt --ckpt-ejecutor cierre={RESULTS_DIR}/sistema_final/maestro/ejecutor_cierre.pt --nombre PPO_sistema 2>&1 | grep -v UserWarning


## Tabla (cuando A y B terminaron)

In [ ]:
# Tabla con las 30 corridas corregidas (solo si los dos cuadernos, A y B, ya terminaron)
import json, os
base = f'{RESULTS_DIR}/corridas_30_completa'
arch = {p: f'{base}/{p}/sistema_meta_ordenes_abides_PPO_sistema_muestreo_impacto_completa.json' for p in ('A', 'B')}
def listas(p):
    if not os.path.exists(arch[p]):
        return 0
    c = json.load(open(arch[p], encoding='utf-8')).get('corridas', {})
    return sum('error' not in v for v in c.values())
n = {p: listas(p) for p in arch}
print('Corridas listas:', n, '(se necesitan 15 en cada una)')
if n['A'] == 15 and n['B'] == 15:
    bench = ' '.join(f'{RESULTS_DIR}/corridas_30/bloque_{k}/benchmarks_meta_ordenes_abides_impacto.json' for k in (1, 2, 3, 4))
    get_ipython().system(f"cd /content/slippage && conda run --no-capture-output -n abides env PYTHONPATH=. python scripts/consolidar_resultados_3_2_2.py --entradas {base}/A {base}/B {bench} --sistema PPO_sistema --out-dir {base}/tabla")
    print(open(f'{base}/tabla/tabla_maestra.md', encoding='utf-8').read())
else:
    print('Falta el otro cuaderno: cuando termine, la tabla y el zip completos salen de ese.')


## Zip para descargar

In [ ]:
import os, zipfile
base = f'{RESULTS_DIR}/corridas_30_completa'
archivos = [os.path.join(c, a) for c, _, fs in os.walk(base) for a in fs if a.endswith(('.json', '.csv', '.md', '.png'))]
zpath = f'{RESULTS_DIR}/corridas_30_completa.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for a in archivos:
        z.write(a, os.path.relpath(a, RESULTS_DIR))
print(len(archivos), 'archivos -> zip listo:', zpath)
